# Build Graph

In [ ]:
from pathlib import Path
from time import perf_counter

import cognee

# Initialization
await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
await cognee.forget(everything=True)

# Data
data_path = "../data/silver"

files = sorted(
    path for path in Path(data_path).rglob("*.md")
    if path.is_file() and path.stat().st_size > 0
)
for path in files:
    print(f"  - {path.relative_to(data_path)}")

# Ingestion
start = perf_counter()
await cognee.remember(
[str(path) for path in files],
dataset_name="docspace",
self_improvement=False,
)
print(f"Ingestion completed {(perf_counter() - start) / 60:.1f} min")

# Ingestion Checks

In [9]:
results = await cognee.recall(
    query_text = "What is the difference between Bitcoin and Ethereum in terms of their technology, use cases, and consensus mechanisms?",
    query_type=cognee.SearchType.GRAPH_COMPLETION,
)
for result in results:
    print(result.text)


2026-11-13T14:28:26.837899 [info     ] Vector collection retrieval completed: Retrieved distances from 7 collections in 0.55s [cognee.shared.logging_utils]

2026-11-13T14:28:26.844185 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]

2026-11-13T14:28:26.864452 [info     ] Vector collection retrieval completed: Retrieved distances from 7 collections in 0.57s [cognee.shared.logging_utils]

2026-11-13T14:28:26.867988 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]

2026-11-13T14:28:27.060076 [info     ] ID-filtered retrieval: 415 nodes and 878 edges in 0.21s [cognee.shared.logging_utils]

2026-11-13T14:28:27.139181 [info     ] Graph projection completed: 415 nodes, 878 edges in 0.08s [CogneeGraph]

2026-11-13T14:28:27.315947 [info     ] ID-filtered retrieval: 415 nodes and 851 edges in 0.45s [cognee.shared.logging_utils]

2026-11-13T14:28:27.372179 [info     ] Graph projection completed: 415 nodes, 851 edges in 0.05s [CogneeGraph]

2026-11-13T14

- Technology
  - Bitcoin: Peer-to-peer electronic cash; blockchain with Merkle-tree transactions; SHA-256 proof-of-work; timestamped, tamper-evident chain; digital signatures.
  - Ethereum: Blockchain platform for programmable smart contracts and decentralized apps (DApps); Ethereum Virtual Machine (EVM) for executing contracts.

- Use cases
  - Bitcoin: Digital cash and value transfer between parties.
  - Ethereum: Programmable contracts and DApps (DeFi, NFTs, tokenized services).

- Consensus mechanisms
  - Bitcoin: Proof-of-Work (PoW) consensus; longest chain with the most work.
  - Ethereum: Proof-of-Stake (PoS) consensus after the Merge; validators stake ETH to propose/validate blocks.


In [11]:
from collections import Counter, defaultdict

from cognee.context_global_variables import set_database_global_context_variables
from cognee.infrastructure.databases.graph import get_graph_engine

# 1. Get ingested documents
datasets = await cognee.datasets.list_datasets()
dataset = next((d for d in datasets if d.name == "docspace"), None)

if dataset is None:
    raise ValueError("Dataset not found: docspace")

documents = await cognee.datasets.list_data(dataset.id)

# 2. Get graph data
async with set_database_global_context_variables(
    dataset.id, dataset.owner_id
):
    graph = await get_graph_engine()
    nodes, edges = await graph.get_graph_data()

node_by_id = {
    str(node_id): props
    for node_id, props in nodes
}

types = Counter(
    props.get("type", "Unknown")
    for _, props in nodes
)

graph_docs = {
    node_id: props
    for node_id, props in node_by_id.items()
    if props.get("type") in ("TextDocument", "Document")
}

chunks = {
    node_id: props
    for node_id, props in node_by_id.items()
    if props.get("type") == "DocumentChunk"
}

chunks_by_doc = defaultdict(set)
parents_by_chunk = defaultdict(set)

for source, target, relation, props in edges:
    source, target = str(source), str(target)

    if source in graph_docs and target in chunks:
        chunks_by_doc[source].add(target)
        parents_by_chunk[target].add(source)

    elif target in graph_docs and source in chunks:
        chunks_by_doc[target].add(source)
        parents_by_chunk[source].add(target)

orphan_chunks = [
    cid for cid in chunks
    if not parents_by_chunk[cid]
]

# 3. Summary
print("=" * 70)
print("Summary for dataset: docspace")
print("=" * 70)

print(f"Local files       : {len(files)}")
print(f"Ingested documents : {len(documents)}")
print(f"Graph documents   : {len(graph_docs)}")
print(f"Graph chunks      : {len(chunks)}")
print(f"Orphan chunks    : {len(orphan_chunks)}")
print(f"Total nodes          : {len(nodes)}")
print(f"Total edges     : {len(edges)}")

print("\nINGESTED DOCUMENTS")
for doc in documents:
    print(f"- {doc.name} | {doc.id}")

print("\nCHUNKS PER GRAPH DOCUMENT")
for doc_id, props in sorted(
    graph_docs.items(),
    key=lambda x: str(x[1].get("name", ""))
):
    name = props.get("name", doc_id)
    count = len(chunks_by_doc[doc_id])
    print(f"- {name}: {count} chunks")

print("\nNODE TYPES")
for node_type, count in types.most_common():
    print(f"- {node_type}: {count}")

print("\nCHECKS")
if files and len(files) != len(documents):
    print("[WARNING] Number of local files differs from dataset")

if len(documents) != len(graph_docs):
    print("[INFO] Number of documents differs between dataset and graph")

if orphan_chunks:
    print(f"[WARNING] {len(orphan_chunks)} chunks without direct link to a document")

if (
    (not files or len(files) == len(documents))
    and len(documents) == len(graph_docs)
    and not orphan_chunks
):
    print("[OK] No discrepancies detected")


2026-11-13T14:32:14.433896 [info     ] Retrieved 415 nodes and 933 edges in 0.05 seconds [cognee.shared.logging_utils]


Summary for dataset: docspace
Local files       : 14
Ingested documents : 14
Graph documents   : 14
Graph chunks      : 52
Orphan chunks    : 0
Total nodes          : 415
Total edges     : 933

INGESTED DOCUMENTS
- BIS bulletin-76 | 31fefbb7-0cf3-43f7-af84-a57f0aa0f47d
- Crypto Assets Monitor | 18a2bb05-0aac-41f7-8272-11e7acf246e4
- Brief remarks on stablecoins | 76b6d6f0-f0c0-4450-9bda-d3bdb63af4be
- Crypto Monitor Q1 2026 | eff6b115-aaee-4e0a-9528-549fab82d947
- 21Shares Report June 2025 | cfbda14e-7d76-4eb1-88ef-62250fe2b70b
- Fact Sheet iShares | 0bc16987-5069-44a4-a25a-42a503fd64ad
- Workshop on Fraud and Investigations Involving Crypto Assets | 1319227e-54ee-4536-b43e-e1c03894ecd7
- GMM Special Feature - Crypto Monitor October 2025 | 5f58b1f3-c2ae-4e4c-a047-19f867ee1ada
- CryptoGlobal_Weekly_2026-07-13 | 8fa495d3-c584-432e-9331-c7064b5add59
- bulletin-125-stablecoin-remuneration-centralised-exchanges | 8678a59b-0689-4dc4-abed-39bf5718dff2
- CryptoGlobal_Weekly_2026-07-11 | 84df84

# Graph Visualization

In [12]:
import webbrowser

await cognee.visualize_graph("graph.html", dataset="docspace", full=True)
webbrowser.open("graph.html")


2026-11-13T14:32:28.405842 [info     ] Retrieved 415 nodes and 933 edges in 0.05 seconds [cognee.shared.logging_utils]

2026-11-13T14:32:28.430866 [info     ] Collected 2 session operation events for the Memory tab. [visualization.session_events]

2026-11-13T14:32:36.040672 [info     ] fetch_node_embeddings: resolved 337/415 node embeddings across 5 collection(s) [embedding_join]

2026-11-13T14:32:37.899080 [info     ] Graph visualization saved as graph.html [cognee.shared.logging_utils]

2026-11-13T14:32:37.903081 [info     ] The HTML file has been stored at path: graph.html [cognee.shared.logging_utils]


True